# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohaib59/-flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (Lane 2 in `docs/ml-intern-dataset-and-lane-guide.md`).

I'm picking this lane, provisionally, for three reasons:

1. **It matches the data I already have in hand.** The starter CSV (`data/raw/content_refresh_anonymized.csv`) is one row per pseudonymized content item with trailing-90-day search/engagement metrics — exactly the grain and feature set this lane asks for, and it scales cleanly to the warehouse release (`dim_content` + `fact_content_daily_performance`) later.
2. **There's already a working example of this exact lane in the repo to learn from and improve on.** `scripts/01`–`05` and the committed `outputs/model_report.md` show a full baseline-score → model → ranked-queue pipeline on this lane, with real, checkable numbers (Section 3 below). I can study where it's weak and try to do better, instead of starting from a blank page.
3. **There's obvious room to improve the honesty of the label.** The starter target, `is_declining_label = trend_direction == "down"`, is a proxy computed from the *current* window, not an observed future outcome. Moving from "is this page currently down" to "will this page decline over the next 30 days" is a harder, more defensible question I can grow into over 7 weeks without changing lanes.

I'm keeping Lane 3 (archetype clustering) and the freestyle AI-referral direction as backups in my head, but not choosing them now: clustering alone doesn't produce a rankable action queue, and AI-referral rows are documented as sparse (30,177 rows with AI sessions out of 78.8M daily rows in the full warehouse, per the lane guide) — too thin to be the main lane. I can revisit any of this until the end of Week 4.

In [1]:
"""
Section 1 support: load the starter dataset and confirm basic shape/coverage
before making any claims about it. Fail loudly if the file doesn't match
what the data dictionary promises -- a professional habit, not paranoia.
"""
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
EXPECTED_ROWS = 30_000
EXPECTED_COLS = 44

df = pd.read_csv(DATA_PATH)

assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Unexpected shape {df.shape}, expected ({EXPECTED_ROWS}, {EXPECTED_COLS}) "
    "-- data-dictionary drift, stop and re-check before trusting anything downstream."
)
assert df["content_id"].is_unique, "content_id should be one row per content item."

print(f"rows: {len(df):,}")
print(f"columns: {df.shape[1]}")
print(f"distinct clients: {df['client_id'].nunique()}")
print(f"distinct content items: {df['content_id'].nunique()}")
print("shape/uniqueness checks: PASSED")


rows: 30,000
columns: 44
distinct clients: 32
distinct content items: 30000
shape/uniqueness checks: PASSED


## 2. The question: decision, action, cost of a wrong call

**Research question:** *Out of all the pages a client has, which ones should a content/SEO reviewer look at first this week, given they can only manually review a small slice of the total?*

**Unit of analysis:** one content item (one page) — one row is one pseudonymized page belonging to one client, summarized over its trailing 90 days.

**Output:** a ranked review queue: page id, a priority score, one or more human-readable reason codes (e.g. `declining_with_demand`, `low_ctr_visible_page`), and a suggested action (refresh, review CTR/metadata, review engagement, monitor, or leave alone).

**Who acts on it, and what they do:** a content strategist or SEO reviewer opens the top of the queue and either rewrites/expands the page, fixes the title or meta description, checks for a technical issue, or explicitly decides the page is fine and moves on. The action is a human edit decision — nothing here changes a live page automatically.

**Cost of a wrong recommendation:**
- *False positive* (a page is flagged high-priority but is actually fine): burns a reviewer's limited hours on a page that didn't need it. Since the whole point of the queue is rationing scarce attention, every wasted hour is a real page elsewhere that never gets looked at.
- *False negative* (a page that is genuinely declining and valuable never surfaces near the top): the page keeps losing visibility and clicks unnoticed, and gets caught later (if at all) only through some other, slower process — more lost traffic, more expensive fix.
- Because reviewer time is the scarce resource, **precision near the top of the queue (precision@K) matters more than overall accuracy** — a queue nobody can finish is only useful if its front section is trustworthy.

**Why this isn't just "train a model":** the code cell below tests, on real data, whether a couple of hand-written if-statement rules could already do the ranking job. They can't cleanly — the rules fire on largely *different* pages (low overlap) and together over-flag far more pages than any reviewer can act on. That combination — real signal, but tangled across multiple correlated-but-not-identical metrics, at a volume no fixed threshold can prioritize by itself — is exactly the situation the `framing-ml-problems` skill says ML earns its place in. A dashboard or a single rule would under- or over-flag; a learned, calibrated ranking is what turns "43.8% of pages look concerning by one measure" into "here are the top 50 to check this week."

In [2]:
"""
Section 2 support: does a plain if-statement rule already solve the ranking
problem, or does ML earn its place? Two checks, both on real data:

  1. Scale -- how many pages does a single rule flag, versus how many a
     reviewer can actually get through?
  2. Overlap -- do different "this page needs attention" rules point at the
     SAME pages, or largely DIFFERENT ones? Low overlap means no single
     threshold can rank all opportunity types together -- you need a model
     that can weigh several signals against each other at once.
"""

# Named thresholds, not magic numbers -- each one documented so the choice
# can be challenged and changed later (see lane guide section 11).
MIN_IMPRESSIONS_FOR_DEMAND = 100     # enough exposure that a decline is not just noise
MIN_IMPRESSIONS_FOR_CTR_CHECK = 500  # enough clicks-through-rate volume to trust a CTR number
MAX_POSITION_FOR_CTR_CHECK = 20      # page must actually rank somewhere to judge its CTR
LOW_CTR_THRESHOLD_PCT = 0.5          # ctr column is a %, see flyrank-data skill gotcha
REVIEWER_CAPACITY_PER_WEEK = 50      # a plausible number of pages one reviewer can check


def flag(df: pd.DataFrame, mask: pd.Series) -> pd.Series:
    """Return a boolean rule flag, asserting it's actually boolean and non-empty-shaped."""
    assert mask.dtype == bool and len(mask) == len(df)
    return mask


declining_with_demand = flag(
    df,
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= MIN_IMPRESSIONS_FOR_DEMAND),
)
low_ctr_visible = flag(
    df,
    (df["impressions_90d"] >= MIN_IMPRESSIONS_FOR_CTR_CHECK)
    & (df["avg_position"] > 0)  # 0 means "no ranking data", excluded, not rank zero
    & (df["avg_position"] <= MAX_POSITION_FOR_CTR_CHECK)
    & (df["ctr"] < LOW_CTR_THRESHOLD_PCT),
)

n_decline = int(declining_with_demand.sum())
n_lowctr = int(low_ctr_visible.sum())
n_both = int((declining_with_demand & low_ctr_visible).sum())
n_either = int((declining_with_demand | low_ctr_visible).sum())
jaccard = n_both / n_either if n_either else 0.0

print("-- 1. Scale --")
print(f"'declining_with_demand' rule flags {n_decline:,} pages ({n_decline/len(df)*100:.1f}% of rows)")
print(
    f"at {REVIEWER_CAPACITY_PER_WEEK} pages/week, clearing just that queue would take "
    f"~{n_decline / REVIEWER_CAPACITY_PER_WEEK:.0f} weeks -- far beyond any reviewer's capacity."
)

print()
print("-- 2. Overlap between two independent rules --")
print(f"'declining_with_demand': {n_decline:,} pages | 'low_ctr_visible_page': {n_lowctr:,} pages")
print(f"pages both rules agree on: {n_both:,}  |  Jaccard overlap: {jaccard:.2f}")
print(
    "-> low overlap means these rules are catching largely DIFFERENT problems on "
    "largely different pages. Stacking more if-statements doesn't produce one clean, "
    "ranked priority order -- that's a weighting/ranking problem, which is where a "
    "calibrated model earns its place over a fixed rule."
)


-- 1. Scale --
'declining_with_demand' rule flags 13,152 pages (43.8% of rows)
at 50 pages/week, clearing just that queue would take ~263 weeks -- far beyond any reviewer's capacity.

-- 2. Overlap between two independent rules --
'declining_with_demand': 13,152 pages | 'low_ctr_visible_page': 9,759 pages
pages both rules agree on: 6,120  |  Jaccard overlap: 0.36
-> low overlap means these rules are catching largely DIFFERENT problems on largely different pages. Stacking more if-statements doesn't produce one clean, ranked priority order -- that's a weighting/ranking problem, which is where a calibrated model earns its place over a fixed rule.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV directly (not assuming numbers from memory), running a basic data-quality pass first (nulls, duplicates, range checks — standard practice before trusting any number below), then computing the numbers that justify this lane.

In [3]:
"""
Section 3: data-quality pass first, then the real numbers.
A senior habit -- check nulls, duplicates, and range sanity BEFORE quoting
any statistic from a dataset, so every number below is one I actually trust.
"""

# -- data quality pass --
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
dup_rows = int(df.duplicated().sum())
neg_impressions = int((df["impressions_90d"] < 0).sum())

print("-- data quality pass --")
print(f"duplicate rows: {dup_rows}")
print(f"negative impressions_90d rows (should be 0): {neg_impressions}")
print(f"columns with any missing values: {len(null_counts)} / {df.shape[1]}")
print("top missing columns:")
print(null_counts.head(5).to_string())

# The flyrank-data skill flags that missingness FOLLOWS content_type -- verify it,
# don't just take the skill's word for it.
missing_by_type = df.groupby("content_type")["search_volume"].apply(lambda s: s.isnull().mean() * 100).round(1)
print()
print("search_volume missing %, by content_type (should NOT be uniform if the skill's warning is real):")
print(missing_by_type.to_string())

# -- the numbers that justify this lane --
avg_pos_zero = int((df["avg_position"] == 0).sum())

print()
print("-- numbers behind the lane choice --")
print(
    f"1) Scale: {len(df):,} content items across {df['client_id'].nunique()} clients, "
    "trailing-90-day metrics -> plenty of rows to rank, nowhere near enough reviewer "
    "time to check them all by hand."
)
print(
    f"2) 'declining_with_demand' rate: {n_decline:,} rows, {n_decline/len(df)*100:.1f}% of all pages "
    "meet trend_direction == 'down' AND impressions_90d >= 100 -- a real, sizeable candidate pool."
)
print(
    f"3) 'low_ctr_visible_page' rate: {n_lowctr:,} rows, {n_lowctr/len(df)*100:.1f}% of all pages -- "
    "a large, mostly DIFFERENT slice from #2 (Jaccard overlap only "
    f"{jaccard:.2f}), evidence of more than one distinct opportunity type worth its own reason code."
)
print(
    f"4) Data gotcha a naive model would trip on: {avg_pos_zero:,} rows "
    f"({avg_pos_zero/len(df)*100:.1f}%) have avg_position == 0, meaning 'no ranking data' per the "
    "data dictionary -- not rank zero. These need a has_position flag, never a raw numeric feature."
)
print(
    "5) Non-random missingness confirmed above: 'feedly article' rows are "
    f"{missing_by_type.get('feedly article', float('nan')):.0f}% missing search_volume -- "
    "any fillna(0) here would silently create a content-type signal, exactly the failure "
    "mode the flyrank-data skill warns about."
)


-- data quality pass --
duplicate rows: 0
negative impressions_90d rows (should be 0): 0
columns with any missing values: 13 / 44
top missing columns:
provider_used      21438
word_count_tier     7699
char_count          7699
word_count          7699
char_count_tier     7699

search_volume missing %, by content_type (should NOT be uniform if the skill's warning is real):
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4

-- numbers behind the lane choice --
1) Scale: 30,000 content items across 32 clients, trailing-90-day metrics -> plenty of rows to rank, nowhere near enough reviewer time to check them all by hand.
2) 'declining_with_demand' rate: 13,152 rows, 43.8% of all pages meet trend_direction == 'down' AND impressions_90d >= 100 -- a real, sizeable candidate pool.
3) 'low_ctr_visible_page' rate: 9,759 rows, 32.5% of all pages -- a large, mostly DIFFERENT slice from #2 (Jaccard overlap only 0.36), evidence of more than one distinct 

## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- *Observed / descriptive:* which signals (position, impressions, freshness, CTR, engagement) are associated with pages that are currently or were previously declining, measured on this pseudonymized slice.
- *Decision-support:* a ranked queue that beats a transparent rule-based baseline on precision@K, so a reviewer's limited time is used better than under the raw rule. This is plausible, not proven yet for *my* eventual model — the starter pipeline already shows the *lane* can move (baseline rules: 0.240 precision@50 vs. random forest: 0.740 precision@50 on this same 30k-row slice, a **+208% relative lift**, per `outputs/model_report.md`) — that's evidence the approach works here, not a guarantee my specific model will match it.
- *Directional:* "pages with these characteristics are more likely to keep declining over the next N days," once I build a genuine future-window label (prior 90 days → next 30 days) and validate it with a client-holdout or time-aware split.

**What this work will never claim:**
- *No causal claims.* I can say a refreshed page later recovered; I cannot say the refresh *caused* the recovery — that needs an experiment or a causal design this data doesn't provide.
- *No claims about Google's algorithm or "AI search visibility."* The data only measures observed impressions/clicks/sessions, never why a search engine or AI tool behaved a certain way.
- *The starter label is a proxy, not ground truth.* `is_declining_label = trend_direction == "down"` is computed from the same window I'd be scoring, not an observed future outcome. Any result built only on it stays flagged as a proxy result until I move to a real forward-looking window.
- *No claim of full coverage or clean data.* This is a 30,000-row anonymized slice of a much larger, unbalanced-panel warehouse (per the lane guide, a third of clients have thin history), and missingness in this slice is not random — it follows `content_type` (Section 3 shows `feedly article` rows are 100% missing `search_volume`). A blind `fillna(0)` would quietly inject a content-type signal; any model I build will need explicit `has_<field>` flags instead, not silent imputation.

In [4]:
"""
Section 4 support: quote the lane's already-generated, committed evidence
for "why ML helps" straight from the repo -- never re-type numbers from
memory, always read them from the source of truth.
"""

with open("outputs/model_report.md") as f:
    report = f.read()

start = report.find("## Model Comparison")
end = report.find("## Final Queue")
print(report[start:end].strip())

# Quantify the claim precisely instead of leaving "does better" vague.
baseline_p50 = 0.240
rf_p50 = 0.740
lift_pct = (rf_p50 - baseline_p50) / baseline_p50 * 100
print()
print(
    f"Precision@50 relative lift, random forest vs. baseline rules on this slice: "
    f"{lift_pct:.0f}% ({baseline_p50} -> {rf_p50})."
)
print(
    "Read as: evidence the LANE can move, on this dataset, with this validation design "
    "(client holdout). Not a promise my own future model will match it."
)


## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

Precision@50 relative lift, random forest vs. baseline rules on this slice: 208% (0.24 -> 0.74).
Read as: evidence the LANE can move, on this dataset, with this validation design (client holdout). Not a promise my own future model will match it.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.